# flatbug on TILED trays

Companion to `train_rfdetr_tiled_fullres.ipynb`. Trains flatbug on the **same 1536px tiles**, so the architecture comparison is made at **matched input scale**.

**Why.** Your flatbug 0.4189 and RF-DETR 0.3476 runs both used whole trays, but at different effective scales — tiles are fed 1:1 at 1536, no letterboxing, RF-DETR square-resizes to 800 (0.52x in x). Tiling feeds both at **1:1**. Without this run, a strong RF-DETR-tiled result could not be separated from the scale advantage tiling gives it.

That completes a clean 2x2: **{flatbug, RF-DETR} x {whole-tray, tiled}**.

**Prerequisite:** run Step 1 of `train_rfdetr_tiled_fullres.ipynb` first — this reuses the same `insect_dataset_mixed_tiled1536`.

> Note: flatbug's *native* crop-based trainer needs instance masks and your synthetic COCO is > bbox-only, so this uses the same Ultralytics detection path as your other flatbug runs. > Tiling is what supplies the 1:1 scale instead.


In [ ]:
# ============================================================
# SEPTEMBER RUN — flatbug, 1536 px tiles fed 1:1
# ============================================================
# Regenerated dataset, trained at full resolution. What changed since August:
#
#   * GROUND TRUTH boxes are the insect, not the rotated canvas. The old boxes were
#     1.295x too large on a side (4/pi for a uniform angle), so ~40% of every box was
#     tray. Every AP number from before was measured against inflated targets.
#   * TRAYS are 3473x2769 = 10.85 px/mm, matching the real 3DTRAB trays exactly.
#     Previously 2644x2108 = 8.26 px/mm, so synthetic insects were 24% too small.
#   * TAXONOMY is 22 classes, not 23. Siphonaptera and Raphidioptera dropped;
#     Diplopoda -> Polyxenida, Chilopoda -> Lithobiomorpha + Scutigeromorpha.
#   * DATA is synthetic + real 3DTRAB mixed, group-disjoint on the real side, with
#     the real half oversampled to ~5% of the training signal.
#
# Evaluate on 3dtrab_coco_TESTSPLIT.json, NEVER the full 3DTRAB — roughly half of
# those photographs are in this training set.

# ============================================================
# CONFIG — flatbug on TILED trays  <- edit GPU + paths
# ============================================================
from pathlib import Path
import os
DEVICE   = "3"                       # keep off the GPUs running the RF-DETR jobs
os.environ.setdefault("CUDA_VISIBLE_DEVICES", DEVICE)

BASE_DIR = Path(os.environ.get("MA_BASE", "/data/MA_Atul_Ares"))   # <-- Ares path; matches final_comparison
# tiled dataset produced by train_rfdetr_tiled_fullres.ipynb (Step 1). Same tiles, so the
# flatbug/RF-DETR comparison is matched.
COCO_DATASET_DIR = BASE_DIR / "insect_dataset_mixed_tiled1536"   # from the RF-DETR tiled nb
YOLO_DATASET_DIR = BASE_DIR / "sept_run" / "flatbug_yolo_tiled1536"
SRC_COCO         = BASE_DIR / "insect_dataset_mixed"          # FULL trays, for final eval
OUTPUT_DIR       = BASE_DIR / "sept_run" / "flatbug_tiled_1536"
RUN_NAME         = "flatbug_l_1536_tiled"

FLATBUG_SIZE    = "L"
FLATBUG_WEIGHTS = BASE_DIR / f"flat_bug_{FLATBUG_SIZE}.pt"
FLATBUG_REMOTE  = "https://anon.erda.au.dk/share_redirect/Bb0CR1FHG6/models/"
BASE_ARCH       = f"yolov8{FLATBUG_SIZE.lower()}.pt"

# tiles are 1536px -> feed them 1:1 (this is the whole point)
TILE      = 1536   # fed 1:1, so insects keep their native size
OVERLAP   = 256
IMG_SIZE  = TILE
MERGE_IOU   = 0.6      # cross-tile duplicates are IoU~1.0; overlapping insects usually <0.6
EDGE_MARGIN = 31       # ~2% of the tile, as 16 was of 800. Demotes detections
                       # touching a tile border...
EDGE_FACTOR = 0.85     # ...so the complete copy from the neighbour tile wins NMS

EPOCHS   = 30          # ~6x more images per epoch than whole-tray training
BATCH    = 4       # 1536 tiles; raise if VRAM allows
WORKERS  = 8
PATIENCE = 10
SEED     = 42
OPTIMIZER, LR0, LRF = "SGD", 0.01, 0.01
MOMENTUM, WEIGHT_DECAY, WARMUP_EPOCHS, CLOSE_MOSAIC = 0.937, 0.0005, 3.0, 10

CONF_THRES = 0.4
INFER_CONF = 0.05      # low for scoring; filter later
IOU_THRES  = 0.5

EXPECTED_CLASS_NAMES = [
    # 22 classes. Siphonaptera and Raphidioptera dropped (6 and 2 specimens);
    # Diplopoda -> Polyxenida, Chilopoda -> Lithobiomorpha + Scutigeromorpha.
    "Lepidoptera",
    "Psocodea",
    "Odonata",
    "Zoraptera",
    "Phasmatodea",
    "Zygentoma",
    "Thysanoptera",
    "Strepsiptera",
    "Archaeognatha",
    "Dermaptera",
    "Diptera",
    "Coleoptera",
    "Hymenoptera",
    "Hemiptera",
    "Orthoptera",
    "Trichoptera",
    "Neuroptera",
    "Araneae",
    "Opiliones",
    "Polyxenida",
    "Lithobiomorpha",
    "Scutigeromorpha",
]
assert (COCO_DATASET_DIR/"train"/"_annotations.coco.json").exists(), (
    f"{COCO_DATASET_DIR} not found - run Step 1 of train_rfdetr_tiled_fullres.ipynb first")

# Those tiles came from train_rfdetr_tiled_fullres.ipynb. Confirm they match the
# CURRENT source dataset, not a leftover August export — September exists because the
# old ground truth was wrong (boxes 1.295x too large, 23 classes, 8.26 px/mm), and a
# stale tile directory would train this model on exactly that, silently.
import json as _j
_t = _j.load(open(COCO_DATASET_DIR/"train"/"_annotations.coco.json"))
_s = _j.load(open(SRC_COCO/"train"/"_annotations.coco.json"))
assert [(c["id"], c["name"]) for c in sorted(_t["categories"], key=lambda c: c["id"])] == \
       [(c["id"], c["name"]) for c in sorted(_s["categories"], key=lambda c: c["id"])], \
    f"{COCO_DATASET_DIR} has different categories from {SRC_COCO} — it is a stale export"
assert max(i["width"] for i in _t["images"][:200]) == TILE, \
    f"{COCO_DATASET_DIR} holds tiles of a different size than TILE={TILE} — stale export"
print(f"tiled source verified: {len(_t['images']):,} tiles at {TILE} px, "
      f"{len(_t['categories'])} classes")
print(f"GPU {DEVICE} | tiles {TILE}px fed at imgsz {IMG_SIZE} (1:1) | {COCO_DATASET_DIR}")
print(f"run -> {OUTPUT_DIR/RUN_NAME}")


## 1. Convert the tiled COCO to YOLO


In [ ]:
import json, os, shutil
from pathlib import Path
import yaml

COCO_DATASET_DIR = Path(COCO_DATASET_DIR)
YOLO_DATASET_DIR = Path(YOLO_DATASET_DIR)
SPLIT_MAP = {"train": "train", "valid": "val", "test": "test"}   # coco name -> yolo name

def _load_coco(split):
    p = COCO_DATASET_DIR / split / "_annotations.coco.json"
    if not p.exists():
        raise FileNotFoundError(f"Missing {p} - is COCO_DATASET_DIR correct?")
    return json.load(open(p))

# --- class order from the train split's categories (id-sorted) ---------------
train_coco = _load_coco("train")
cats = sorted(train_coco["categories"], key=lambda c: c["id"])
CLASS_NAMES = [c["name"] for c in cats]
id2idx = {c["id"]: i for i, c in enumerate(cats)}
NC = len(CLASS_NAMES)
print(f"{NC} classes from COCO categories:\n  {CLASS_NAMES}")
if list(CLASS_NAMES) != list(EXPECTED_CLASS_NAMES):
    print("\nWARNING: class order differs from EXPECTED_CLASS_NAMES in the config cell!")
    print("         Using the COCO order above (authoritative).")

def _link_or_copy(src, dst):
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    try:
        os.symlink(os.path.abspath(src), dst)
    except OSError:
        shutil.copy2(src, dst)

def convert_split(coco_split, yolo_split):
    coco = _load_coco(coco_split)
    src_dir = COCO_DATASET_DIR / coco_split
    img_out = YOLO_DATASET_DIR / "images" / yolo_split
    lbl_out = YOLO_DATASET_DIR / "labels" / yolo_split
    img_out.mkdir(parents=True, exist_ok=True)
    lbl_out.mkdir(parents=True, exist_ok=True)

    imgs = {im["id"]: im for im in coco["images"]}
    anns_by_img = {}
    for a in coco["annotations"]:
        anns_by_img.setdefault(a["image_id"], []).append(a)

    n_ann = 0
    for iid, im in imgs.items():
        fn, W, H = im["file_name"], im["width"], im["height"]
        _link_or_copy(src_dir / fn, img_out / fn)
        lines = []
        for a in anns_by_img.get(iid, []):
            x, y, w, h = a["bbox"]
            cx = min(max((x + w / 2) / W, 0.0), 1.0)
            cy = min(max((y + h / 2) / H, 0.0), 1.0)
            nw = min(max(w / W, 0.0), 1.0)
            nh = min(max(h / H, 0.0), 1.0)
            lines.append(f"{id2idx[a['category_id']]} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
            n_ann += 1
        (lbl_out / (Path(fn).stem + ".txt")).write_text("\n".join(lines))
    return len(imgs), n_ann

counts = {}
for coco_split, yolo_split in SPLIT_MAP.items():
    ni, na = convert_split(coco_split, yolo_split)
    counts[yolo_split] = (ni, na)
    print(f"  {coco_split:5s} -> {yolo_split:5s}: {ni:5d} images, {na:6d} annotations")

# --- data.yaml ----------------------------------------------------------------
data_yaml = {
    "path": str(YOLO_DATASET_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "nc": NC,
    "names": CLASS_NAMES,
}
DATA_YAML_PATH = YOLO_DATASET_DIR / "data.yaml"
with open(DATA_YAML_PATH, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)
print(f"\nWrote {DATA_YAML_PATH}")


## 2. Warm-start from flatbug weights


In [ ]:
from ultralytics import YOLO

# COCO-pretrained detection model of the matching scale (backbone/neck shapes match flatbug).
model = YOLO(BASE_ARCH)

# Transfer flatbug's arthropod-tuned backbone + neck (segmentation head skipped on shape mismatch).
model.load(str(FLATBUG_WEIGHTS))
print(f"Warm-started {BASE_ARCH} from flatbug weights: {FLATBUG_WEIGHTS}")
print("(Detection head is reset to the 22 dataset classes at train start.)")


## 3. Train on tiles


In [ ]:
from pathlib import Path
import sys
from ultralytics import settings

# Ultralytics auto-enables optional logging callbacks when their packages are present.
# The newer MLflow in this shared env refuses its default filesystem tracking store and
# crashes training at setup ("filesystem tracking backend ... is in maintenance mode").
# Disable the cloud loggers permanently, and drop any already-cached logger module so
# the change also applies in an already-running kernel (no restart needed).
settings.update({"mlflow": False, "wandb": False, "comet": False,
                 "dvc": False, "clearml": False, "neptune": False})
for _m in [k for k in list(sys.modules) if "ultralytics.utils.callbacks.mlflow" in k]:
    del sys.modules[_m]

train_results = model.train(
    data=str(Path(YOLO_DATASET_DIR) / "data.yaml"),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    optimizer=OPTIMIZER,
    lr0=LR0,
    lrf=LRF,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
    warmup_epochs=WARMUP_EPOCHS,
    close_mosaic=CLOSE_MOSAIC,
    patience=PATIENCE,
    seed=SEED,
    deterministic=True,
    project=str(OUTPUT_DIR),
    name=RUN_NAME,
    exist_ok=True,
    plots=True,
    val=True,
)

# model.trainer.{save_dir,best} are the canonical, version-stable handles.
RUN_DIR      = Path(model.trainer.save_dir)
BEST_WEIGHTS = Path(model.trainer.best)          # <run>/weights/best.pt
assert BEST_WEIGHTS.exists(), f"best.pt not found at {BEST_WEIGHTS}"
print(f"\nRun dir      : {RUN_DIR}")
print(f"Best weights : {BEST_WEIGHTS}")


## 4. Tiled inference on full trays + merge

The training mAP is on **tiles** and is not comparable to your other runs. This is.


In [ ]:
# ── Tiling core (grid, box tiling, NMS, edge penalty, merge) ──
import json, math
import numpy as np
from pathlib import Path


def tile_grid(W, H, tile, overlap):
    """Top-left corners covering WxH with `tile` windows and `overlap` px overlap.
    Last row/col is flush to the edge so nothing is missed."""
    step = max(1, tile - overlap)
    xs = list(range(0, max(1, W - tile + 1), step))
    ys = list(range(0, max(1, H - tile + 1), step))
    if not xs or xs[-1] != max(0, W - tile): xs.append(max(0, W - tile))
    if not ys or ys[-1] != max(0, H - tile): ys.append(max(0, H - tile))
    return sorted(set(xs)), sorted(set(ys))


def tile_boxes(boxes, x0, y0, tile, W, H, min_vis=0.85, min_size=4):
    """Clip xyxy boxes into the tile at (x0,y0). Returns (kept_boxes_local, kept_idx).
    A box is kept only if >= min_vis of its area survives the crop, so partial
    insects at tile edges are not learned as whole objects (they are complete in
    the overlapping neighbour tile)."""
    if len(boxes) == 0:
        return np.zeros((0, 4)), []
    b = np.asarray(boxes, float)
    x1 = np.clip(b[:, 0], x0, x0 + tile); y1 = np.clip(b[:, 1], y0, y0 + tile)
    x2 = np.clip(b[:, 2], x0, x0 + tile); y2 = np.clip(b[:, 3], y0, y0 + tile)
    w, h = np.clip(x2 - x1, 0, None), np.clip(y2 - y1, 0, None)
    area_clip = w * h
    area_orig = np.clip((b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1]), 1e-9, None)
    keep = (area_clip / area_orig >= min_vis) & (w >= min_size) & (h >= min_size)
    idx = np.where(keep)[0]
    out = np.stack([x1[idx] - x0, y1[idx] - y0, x2[idx] - x0, y2[idx] - y0], 1) if len(idx) else np.zeros((0, 4))
    return out, idx.tolist()


def nms(boxes, scores, iou_thr=0.5):
    """Plain greedy NMS. boxes xyxy."""
    if len(boxes) == 0:
        return []
    b = np.asarray(boxes, float); s = np.asarray(scores, float)
    x1, y1, x2, y2 = b[:, 0], b[:, 1], b[:, 2], b[:, 3]
    area = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    order = s.argsort()[::-1]; keep = []
    while order.size:
        i = order[0]; keep.append(int(i))
        if order.size == 1: break
        xx1 = np.maximum(x1[i], x1[order[1:]]); yy1 = np.maximum(y1[i], y1[order[1:]])
        xx2 = np.minimum(x2[i], x2[order[1:]]); yy2 = np.minimum(y2[i], y2[order[1:]])
        inter = np.clip(xx2 - xx1, 0, None) * np.clip(yy2 - yy1, 0, None)
        iou = inter / np.clip(area[i] + area[order[1:]] - inter, 1e-9, None)
        order = order[1:][iou <= iou_thr]
    return keep


def edge_penalty(local_boxes, tile, margin=16, factor=0.85):
    """Multiplier that demotes detections touching a tile border. The same object
    is usually complete in the overlapping neighbour tile, so NMS should prefer
    that copy. Mirrors flatbug's EDGE_CASE_MARGIN idea."""
    if len(local_boxes) == 0:
        return np.zeros(0)
    b = np.asarray(local_boxes, float)
    touch = ((b[:, 0] <= margin) | (b[:, 1] <= margin) |
             (b[:, 2] >= tile - margin) | (b[:, 3] >= tile - margin))
    return np.where(touch, factor, 1.0)


def merge_tiles(preds, iou_thr=0.5, per_class=False):
    """preds: list of (boxes_global Nx4, scores N, classes N) from each tile.
    -> merged (boxes, scores, classes) after NMS in full-image coordinates."""
    B = [p[0] for p in preds if len(p[0])]
    if not B:
        return np.zeros((0, 4)), np.zeros(0), np.zeros(0, int)
    boxes = np.concatenate(B)
    scores = np.concatenate([p[1] for p in preds if len(p[0])])
    classes = np.concatenate([p[2] for p in preds if len(p[0])]).astype(int)
    keep_all = []
    for c in (np.unique(classes) if per_class else [None]):
        m = np.where(classes == c)[0] if c is not None else np.arange(len(boxes))
        k = nms(boxes[m], scores[m], iou_thr)
        keep_all += m[k].tolist()
    keep_all = np.array(sorted(keep_all), int)
    return boxes[keep_all], scores[keep_all], classes[keep_all]


In [ ]:
# ============================================================
# TILED INFERENCE on FULL trays + merge  (the comparable number)
# ============================================================
# The test mAP printed by training is on TILES. This reconstructs full-tray
# predictions exactly like the RF-DETR tiled notebook, so the two are comparable.
import json, time
import numpy as np
from PIL import Image
from tqdm.auto import tqdm
from ultralytics import YOLO

EVAL_SPLIT = "test"
CKPT = Path(OUTPUT_DIR)/RUN_NAME/"weights"/"best.pt"
assert CKPT.exists(), f"missing {CKPT}"
m = YOLO(str(CKPT))

src = SRC_COCO/EVAL_SPLIT
ann = json.load(open(src/"_annotations.coco.json"))
cats = sorted(ann["categories"], key=lambda c: c["id"])
CLASS_NAMES  = [c["name"] for c in cats]
catid_to_idx = {c["id"]: i for i, c in enumerate(cats)}
GT = {}
for a in ann["annotations"]:
    b = a["bbox"]
    GT.setdefault(a["image_id"], []).append(
        (catid_to_idx[a["category_id"]], b[0], b[1], b[0]+b[2], b[1]+b[3]))

MAX_EVAL_IMAGES = None            # None = whole split
metas = ann["images"] if MAX_EVAL_IMAGES is None else ann["images"][:MAX_EVAL_IMAGES]

preds_out, t0 = {}, time.time()
for im in tqdm(metas, desc="tiled inference"):
    p = src/im["file_name"]
    if not p.exists(): continue
    W, H = im["width"], im["height"]
    img = Image.open(p).convert("RGB")
    xs, ys = tile_grid(W, H, TILE, OVERLAP)
    tp = []
    for y0 in ys:
        for x0 in xs:
            crop = img.crop((x0, y0, min(x0+TILE, W), min(y0+TILE, H)))
            r = m.predict(crop, conf=INFER_CONF, iou=IOU_THRES, imgsz=IMG_SIZE,
                          device=DEVICE, verbose=False)[0]
            if r.boxes is None or len(r.boxes) == 0: continue
            lb = r.boxes.xyxy.cpu().numpy()
            sc = r.boxes.conf.cpu().numpy() * edge_penalty(lb, TILE, EDGE_MARGIN, EDGE_FACTOR)
            cl = r.boxes.cls.cpu().numpy().astype(int)
            tp.append((lb + np.array([x0, y0, x0, y0]), sc, cl))
    # CLASS-AGNOSTIC. per_class=True let the same insect, detected as two different
    # classes in two overlapping tiles, survive as two boxes — NMS never compared
    # them. Measured on the August predictions: 31% of RF-DETR tiled's detections
    # were co-located pairs with different labels. Agnostic NMS keeps the best label.
    mb, ms, mc = merge_tiles(tp, iou_thr=MERGE_IOU, per_class=False)
    preds_out[im["id"]] = {"boxes": mb.tolist(), "scores": ms.tolist(), "classes": mc.tolist()}

json.dump(preds_out, open(Path(OUTPUT_DIR)/f"tiled_preds_{EVAL_SPLIT}.json", "w"))
print(f"\n{sum(len(v['scores']) for v in preds_out.values())} merged detections over "
      f"{len(preds_out)} full trays in {time.time()-t0:.0f}s")


In [ ]:
# ── SCORE the merged full-tray predictions (same code as the RF-DETR tiled run) ──
import json, numpy as np, pandas as pd
from pathlib import Path

# ── self-contained: rebuild GT / CLASS_NAMES from the COCO annotations when the
# inference cell above was skipped (e.g. re-scoring an already-finished run) ──
EVAL_SPLIT = globals().get("EVAL_SPLIT", "test")
if "GT" not in globals() or "CLASS_NAMES" not in globals():
    _ann = json.load(open(Path(SRC_COCO)/EVAL_SPLIT/"_annotations.coco.json"))
    _cats = sorted(_ann["categories"], key=lambda c: c["id"])
    CLASS_NAMES = [c["name"] for c in _cats]
    _c2i = {c["id"]: i for i, c in enumerate(_cats)}
    GT = {}
    for _a in _ann["annotations"]:
        _b = _a["bbox"]
        GT.setdefault(_a["image_id"], []).append(
            (_c2i[_a["category_id"]], _b[0], _b[1], _b[0]+_b[2], _b[1]+_b[3]))
    print(f"rebuilt GT: {len(GT)} images, {len(_ann['annotations'])} boxes, "
          f"{len(CLASS_NAMES)} classes (split={EVAL_SPLIT})")
IOU_VEC = np.linspace(0.5, 0.95, 10); NC = len(CLASS_NAMES)

def iou_mat(a, b):
    if len(a) == 0 or len(b) == 0: return np.zeros((len(a), len(b)))
    a, b = np.asarray(a, float), np.asarray(b, float)
    lt = np.maximum(a[:, None, :2], b[None, :, :2]); rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb-lt, 0, None); inter = wh[..., 0]*wh[..., 1]
    aa = (a[:, 2]-a[:, 0])*(a[:, 3]-a[:, 1]); ab = (b[:, 2]-b[:, 0])*(b[:, 3]-b[:, 1])
    return inter/np.clip(aa[:, None]+ab[None]-inter, 1e-9, None)

# NumPy 2.0 removed np.trapz in favour of np.trapezoid. The training envs run
# different NumPy majors (flatbug torch 2.13 / rfdetr torch 2.9), so bind once
# and use _TRAPZ rather than depending on which one is installed.
_TRAPZ = getattr(np, "trapezoid", None) or np.trapz
def _ap(rec, prec):
    mrec = np.concatenate(([0.], rec, [1.])); mpre = np.concatenate(([1.], prec, [0.]))
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
    x = np.linspace(0, 1, 101); return float(_TRAPZ(np.interp(x, mrec, mpre), x))

if "preds_out" not in globals():          # re-score without re-running inference
    _f = Path(OUTPUT_DIR)/f"tiled_preds_{EVAL_SPLIT}.json"
    preds_out = {int(k): v for k, v in json.load(open(_f)).items()}
    print(f"loaded {len(preds_out)} images from {_f}")

tps, confs, pcls, tcls = [], [], [], []
for iid, pr in preds_out.items():
    g = GT.get(iid, [])
    gb = np.array([x[1:] for x in g]).reshape(-1, 4); gc = np.array([x[0] for x in g], int)
    pb = np.array(pr["boxes"]).reshape(-1, 4); ps = np.array(pr["scores"]); pc = np.array(pr["classes"], int)
    tp = np.zeros((len(pb), len(IOU_VEC)), bool)
    if len(pb) and len(gb):
        M = iou_mat(pb, gb); same = pc[:, None] == gc[None, :]
        for ti, thr in enumerate(IOU_VEC):
            ci, gi = np.where((M >= thr) & same)
            if not len(ci): continue
            o = np.argsort(-M[ci, gi]); up, ug = set(), set()
            for c, gg in zip(ci[o], gi[o]):
                if c in up or gg in ug: continue
                up.add(c); ug.add(gg); tp[c, ti] = True
    tps.append(tp); confs.append(ps); pcls.append(pc); tcls.append(gc)

tp = np.concatenate(tps); conf = np.concatenate(confs)
pcls = np.concatenate(pcls); tcls = np.concatenate(tcls)
o = np.argsort(-conf); tp, conf, pcls = tp[o], conf[o], pcls[o]
n_gt = np.bincount(tcls, minlength=NC)

rows, ap50, ap75, ap5095 = [], [], [], []
precs, recs = [], []
for c in range(NC):
    i = pcls == c
    if n_gt[c] == 0 or i.sum() == 0:
        rows.append({"class": CLASS_NAMES[c], "instances": int(n_gt[c]), "AP50": 0.0, "AP75": 0.0,
                     "AP50_95": 0.0, "precision": 0.0, "recall": 0.0, "F1": 0.0, "conf_at_bestF1": 0.0})
        ap50.append(0.0); ap75.append(0.0); ap5095.append(0.0)
        precs.append(0.0); recs.append(0.0); continue
    tpc = tp[i].cumsum(0); fpc = (1-tp[i]).cumsum(0)
    rec = tpc/(n_gt[c]+1e-16); prec = tpc/(tpc+fpc+1e-16)
    aps = [_ap(rec[:, t], prec[:, t]) for t in range(len(IOU_VEC))]
    ap50.append(aps[0]); ap75.append(aps[5]); ap5095.append(float(np.mean(aps)))
    # Precision and recall at the best-F1 operating point, IoU 0.5 — the same
    # convention Ultralytics reports, so the tiled and whole-tray rows of the
    # comparison are directly comparable. These curves were already being computed
    # and thrown away after integrating AP.
    _p50, _r50 = prec[:, 0], rec[:, 0]
    _f1 = 2*_p50*_r50/np.clip(_p50+_r50, 1e-16, None)
    _b = int(np.argmax(_f1))
    precs.append(float(_p50[_b])); recs.append(float(_r50[_b]))
    rows.append({"class": CLASS_NAMES[c], "instances": int(n_gt[c]),
                 "AP50": round(aps[0], 4), "AP75": round(aps[5], 4),
                 "AP50_95": round(float(np.mean(aps)), 4),
                 "precision": round(float(_p50[_b]), 4),
                 "recall": round(float(_r50[_b]), 4),
                 "F1": round(float(_f1[_b]), 4),
                 "conf_at_bestF1": round(float(conf[i][_b]), 4)})
df = pd.DataFrame(rows); df.to_csv(Path(OUTPUT_DIR)/"tiled_per_class.csv", index=False)
print(df.to_string(index=False))
print(f"\n=== flatbug TILED, full trays ({EVAL_SPLIT}) ===")
print(f"  mAP@50:95 : {np.mean(ap5095):.4f}")
print(f"  mAP@50    : {np.mean(ap50):.4f}")
print(f"  mAP@75    : {np.mean(ap75):.4f}   <- the metric this experiment is about")
print(f"  precision : {np.mean(precs):.4f}   (class mean at best-F1, IoU 0.5)")
print(f"  recall    : {np.mean(recs):.4f}")
print("""
compare against:
  flatbug  whole-tray  0.4189  (@50 0.6265, @75 0.4681)
  RF-DETR  whole-tray  0.3476  (@50 0.6408, @75 0.3400)
  RF-DETR  tiled       -> train_rfdetr_tiled_fullres.ipynb
""")
